# T7SL — full-frame annotated MP4 review
This run reads completed8-video measurements and counting results. It performs no new detector, tracker, ReID or counting. CPU rendering only. Each source frame is verified and written to H264; encoded output is fully decoded for verification.

In [ ]:
from pathlib import Path
import os,sys,json,subprocess
ROOT=Path('/kaggle/working/t7sl_annotation_source');ROOT.mkdir(exist_ok=False);os.chdir(ROOT)


In [ ]:
for name in ['src/traffic_location', 'src/traffic_location', 'src/traffic_location/traffic', 'src/traffic_location/traffic', 'src/traffic_location/traffic', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'tools']:
 (ROOT/name).mkdir(parents=True,exist_ok=True)


## src/traffic_location/__init__.py

In [ ]:
%%writefile /kaggle/working/t7sl_annotation_source/src/traffic_location/__init__.py
"""Versioned engine boundaries. No inference, network or filesystem work on import."""
__version__ = '0.1.0'


## src/traffic_location/artifacts.py

In [ ]:
%%writefile /kaggle/working/t7sl_annotation_source/src/traffic_location/artifacts.py
"""Local immutable artifact boundaries; storage vendor adapters remain external."""
import gzip,hashlib,json
from pathlib import Path

def canonical(value):return json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=False,allow_nan=False).encode()
def semantic_hash(value):return hashlib.sha256(canonical(value)).hexdigest()
def sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
 return h.hexdigest()
def require(test,message):
 if not test:raise ValueError(message)
def artifact(ref,root):
 require(isinstance(ref,dict) and set(ref)>={'path','sha256'},'Explicit artifact path and SHA256 required')
 p=Path(ref['path']);p=Path(root)/p if not p.is_absolute() else p
 require(not p.is_symlink(),'Symlink artifact is not supported')
 p=p.resolve()
 require(p.is_file(),'Artifact missing/unsafe')
 require(isinstance(ref['sha256'],str) and len(ref['sha256'])==64 and sha(p)==ref['sha256'],'Artifact checksum differs: '+p.name)
 return p
def read(path):
 data=Path(path).read_bytes()
 return json.loads(gzip.decompress(data) if str(path).endswith('.gz') else data)
def write(path,value):
 path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);data=canonical(value)
 path.write_bytes(gzip.compress(data,mtime=0) if path.suffix=='.gz' else data)


## src/traffic_location/traffic/__init__.py

In [ ]:
%%writefile /kaggle/working/t7sl_annotation_source/src/traffic_location/traffic/__init__.py
"""Independent engine components; legacy algorithm semantics are preserved."""


## src/traffic_location/traffic/results.py

In [ ]:
%%writefile /kaggle/working/t7sl_annotation_source/src/traffic_location/traffic/results.py
"""Domain reconciliation and stable result identities, separate from native ledgers."""
from collections import Counter
from copy import deepcopy
import platform
from ..artifacts import require,semantic_hash

def validate_native(result):
 a=result['aggregate'];events=result['events']
 require(len(events)==a['total'],'Accepted total/event mismatch')
 require(len({e['event_id'] for e in events})==len(events),'Duplicate accepted event ID')
 measured=sum(e['evidence']['measured_crossing'] is True for e in events)
 require(measured==a['measured_total'] and len(events)-measured==a['inferred_total'],'Measured/estimated mismatch')
 require(dict(sorted(Counter(e['category'] for e in events).items()))==a['by_category'],'Category aggregation mismatch')
 require(dict(sorted(Counter(e['direction'] for e in events).items()))==a['by_direction'],'Direction aggregation mismatch')
 require(all(e.get('accepted_count_event') is True and e.get('status')=='accepted' for e in events),'Non-accepted event in accepted ledger')
 require(sum(d.get('status')=='pending' for d in result['decisions'])==a['pending'],'Pending decision aggregate mismatch')
 require(len(result['invalid_decisions'])==a['invalid'],'Invalid decision aggregate mismatch')
 return result

def wrap_native(result,*,request,branch,implementation_hash):
 validate_native(result);a=result['aggregate']
 runtime={'python':platform.python_version(),'implementation':platform.python_implementation()}
 fingerprint=semantic_hash({'source':request['source_identity'],'artifacts':{k:v['sha256'] for k,v in request['artifacts'].items()},
  'branch':branch,'video_id':request['video_id'],'frame_count':request['frame_count'],'implementation':implementation_hash,'runtime':runtime})
 return {'schema_version':'traffic_result_v1','execution_status':'SUCCEEDED',
  'result_quality':'PARTIAL','validation_level':'TECHNICAL_PILOT',
  'result_version':fingerprint,'branch':branch,'operation_profile':request['profile'],'runtime':runtime,
  'scope':{'video_id':request['video_id'],'source_frames':request['frame_count'],'processed_frames':result['report']['frame_count']},
  'summary':{'accepted_total':a['total'],'accepted_measured':a['measured_total'],'accepted_estimated':a['inferred_total'],
   'pending':a['pending'],'invalid':a['invalid'],'native_aggregate':deepcopy(a)},
  'source_identity':deepcopy(request['source_identity']),'native_semantic_sha256':semantic_hash(result),
  'legacy_baseline_review_context_present':'baseline_reviews' in request['artifacts'],
  'fresh_inference_executed':request.get('measurement_format')=='fresh_v1','production_ready':False,
  'limitations':['Counting correctness and identity/role generalization are not established by modular parity',
    'Machine native gate remains unresolved; review-assisted is not automatic model performance']}


## src/traffic_location/traffic/annotation.py

In [ ]:
%%writefile /kaggle/working/t7sl_annotation_source/src/traffic_location/traffic/annotation.py
"""Fresh full-frame audit rendering. Consumes immutable decisions, never counts."""
from collections import defaultdict,Counter
from pathlib import Path
import hashlib,subprocess,math,json
from ..artifacts import read,sha,semantic_hash,require,write
from .results import validate_native
from ._kernel.geometry import Geometry
COLORS={'motorcycle':(70,210,70),'car':(220,100,40),'truck':(30,150,250),'bus':(190,60,180),'pedestrian':(220,210,30),'person':(140,140,140)}
CATS=('motorcycle','car','truck','bus','pedestrian')
def plan(native):
 validate_native(native);by_frame=defaultdict(dict);links=defaultdict(dict);candidates=defaultdict(lambda:defaultdict(set));events=[];display=defaultdict(list)
 for a in native.get('frame_assignments',[]):by_frame[a['frame_index']][a['track_id']]=a
 for r in native.get('component_links',[]):
  for f in r['support_frames']:candidates[f][r['component_track_id']].add(r['vehicle_track_id'])
 for f,children in candidates.items():
  for child,parents in children.items():
   if len(parents)==1:links[f][child]=next(iter(parents))
 for e in native['events']:
  frame=e.get('activation_frame',e['decision_frame']);require(type(frame) is int and 0<=frame<native['report']['frame_count'],'Accepted event activation outside source')
  events.append((frame,e))
  for ref in e['evidence'].get('source_refs',[]):
   ep=ref.get('physical_episode_id')
   if ep:display[ref['track_id'],ep].append((frame,e))
 return {'assignments':by_frame,'links':links,'events':sorted(events,key=lambda v:(v[0],v[1]['event_id'])),'display':display}
def snapshot(p,index):
 active=[e for f,e in p['events'] if f<=index];pivot={c:{d:{'measured':0,'estimated':0} for d in ['A_TO_B','B_TO_A']} for c in CATS}
 for e in active:pivot[e['category']][e['direction']]['measured' if e['evidence']['measured_crossing'] else 'estimated']+=1
 return {'accepted':len(active),'vehicles':sum(e['category']!='pedestrian' for e in active),'pedestrian':sum(e['category']=='pedestrian' for e in active),'measured':sum(e['evidence']['measured_crossing'] is True for e in active),'estimated':sum(e['evidence']['measured_crossing'] is not True for e in active),'pivot':pivot,'recent':active[-7:]}
def paint(image,record,native,geometry,p,*,fps,video_id):
 import cv2,numpy as np
 index=record['frame_index'];h,w=image.shape[:2];canvas=np.zeros((max(900,h),w+560,3),dtype=np.uint8);canvas[:h,:w]=image
 assignments=p['assignments'].get(index,{});names={int(k):v for k,v in geometry['_class_names'].items()};obs={v['track_id']:v for v in record['emitted_measurements'] if names[v['raw_detection']['class_id']] in {'person','motorcycle','car','truck','bus'}};children=defaultdict(list);occupied=[]
 for child,parent in p['links'].get(index,{}).items():
  if child in obs and parent in obs and assignments.get(child,{}).get('role')=='rider' and names[obs[parent]['raw_detection']['class_id']]=='motorcycle':children[parent].append(child)
 hidden={c for cs in children.values() for c in cs}
 overlay=canvas[:h,:w].copy()
 for key,color in [('side_a_polygon',(180,80,20)),('side_b_polygon',(30,100,220))]:
  pts=np.array([[round(x*w),round(y*h)] for x,y in geometry['normalized'][key]],dtype=np.int32);cv2.fillPoly(overlay,[pts],color)
 canvas[:h,:w]=cv2.addWeighted(overlay,.15,canvas[:h,:w],.85,0)
 a,b=[np.array(geometry['normalized']['line'][k])*[w,h] for k in ['P1','P2']];normal=np.array([-(b-a)[1],(b-a)[0]])/np.linalg.norm(b-a)
 cv2.line(canvas,tuple(a.astype(int)),tuple(b.astype(int)),(0,255,255),2)
 margin=native['report']['policy']['line_margin_half_width_px']
 for sign in [-1,1]:cv2.line(canvas,tuple((a+sign*margin*normal).astype(int)),tuple((b+sign*margin*normal).astype(int)),(0,150,160),1)
 for tid,o in obs.items():
  if tid in hidden:continue
  raw=o['raw_detection'];cat=names[raw['class_id']];assignment=assignments.get(tid,{});role=assignment.get('role','unknown_person' if cat=='person' else 'vehicle');category='pedestrian' if role=='pedestrian' else cat
  box=list(raw['bbox']);bc=((box[0]+box[2])/2,box[3])
  for child in children.get(tid,[]):
   cb=obs[child]['raw_detection']['bbox'];box=[min(box[0],cb[0]),min(box[1],cb[1]),max(box[2],cb[2]),max(box[3],cb[3])]
  color=COLORS.get(category,(130,130,130));x1,y1,x2,y2=[int(v) for v in box];cv2.rectangle(canvas,(x1,y1),(x2,y2),color,2);cv2.circle(canvas,tuple(int(v) for v in bc),4,(0,255,255),-1)
  label=f'RAW {tid} {category.upper()}'
  if cat=='person':label+=' '+role.upper()
  if children.get(tid):label+=' +R'+','.join(str(c) for c in children[tid])
  candidates=[e for frame,e in p['display'].get((tid,assignment.get('physical_episode_id')),[]) if frame<=index]
  if candidates:
   accepted=max(candidates,key=lambda e:e['decision_frame']);label+=' FINAL '+accepted['category'].upper()+' '+accepted['display_id']
  tw,th=cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,.42,1)[0];lx=max(0,min(x1,w-tw-6));ly=max(th+5,y1-4)
  for candidate_y in [ly]+list(range(th+8,h-5,th+10)):
   rect=(lx,candidate_y-th-5,min(w,lx+tw+6),candidate_y+3)
   if not any(rect[0]<r[2] and r[0]<rect[2] and rect[1]<r[3] and r[1]<rect[3] for r in occupied):ly=candidate_y;break
  occupied.append((lx,ly-th-5,min(w,lx+tw+6),ly+3))
  cv2.rectangle(canvas,(lx,ly-th-5),(min(w,lx+tw+6),ly+3),color,-1);cv2.putText(canvas,label,(lx+3,ly),0,.42,(0,0,0) if category in ['motorcycle','truck','pedestrian'] else (255,255,255),1,cv2.LINE_AA)
 s=snapshot(p,index)
 def text(msg,y,color=(230,230,230),scale=.5,x=None):cv2.putText(canvas,str(msg),(w+12 if x is None else x,y),0,scale,color,1,cv2.LINE_AA)
 text(video_id,25,scale=.53);text(f'FRAME {index} / {native["report"]["frame_count"]-1} | t~{index/fps:.2f}s',49)
 text('OFFLINE FINALIZED REVIEW',74,(0,220,255));text('Uses full-video evidence; not live decisions',96,scale=.43)
 text('PASSAGE ACCEPTED (through this frame)',128,(100,240,100));text(f'Vehicles {s["vehicles"]} | Pedestrian {s["pedestrian"]}',151)
 text(f'Measured {s["measured"]} | Estimated {s["estimated"]}',174)
 text('CATEGORY        A>B M/E       B>A M/E',207,(0,220,255))
 for n,cat in enumerate(CATS):
  v=s['pivot'][cat];text(f'{cat:12} {v["A_TO_B"]["measured"]:4}/{v["A_TO_B"]["estimated"]:<4}    {v["B_TO_A"]["measured"]:4}/{v["B_TO_A"]["estimated"]:<4}',232+n*25,COLORS[cat],.48)
 text('PASSAGE LOG (accepted only)',380,(0,220,255))
 for n,e in enumerate(reversed(s['recent'])):
  text(f'{e["display_id"]} RAW{e["track_id"]} {e["category"]} {e["direction"]}',403+n*39,scale=.43)
  text(f'  f{e["decision_frame"]} {e["route"]} '+('MEASURED' if e['evidence']['measured_crossing'] else 'ESTIMATED'),419+n*39,scale=.39)
 text(f'PENDING FINAL: {native["aggregate"]["pending"]} (whole video)',702,(60,190,255))
 pending=[d for d in native['decisions'] if d.get('status')=='pending']
 for n,d in enumerate(pending[:3]):text(f'RAW{d.get("track_id")} '+','.join(d.get('reasons',[]))[:56],725+n*22,scale=.37)
 text(f'INVALID FINAL: {native["aggregate"]["invalid"]} | excluded',812,(80,130,255))
 text('Pending/invalid never enter accepted pivot',838,scale=.43)
 text('Geometry provisional | source BC: yellow dot',862,scale=.42)
 text('UNITS ON FRAME (raw IDs; rider parent exact-frame)',h+25,x=12,scale=.45)
 for n,(tid,o) in enumerate(list(obs.items())[:11]):
  row=assignments.get(tid,{});par=p['links'].get(index,{}).get(tid);line=f'RAW{tid}: {names[o["raw_detection"]["class_id"]]} | {row.get("role","unresolved")}'
  if par is not None:line+=f' | parent RAW{par}'
  text(line,h+48+n*23,x=12,scale=.42)
 return canvas,s

def render(video,measurement,native_path,geometry_path,output,*,preview_limit=None):
 import cv2
 video,measurement,native_path,geometry_path,output=map(Path,[video,measurement,native_path,geometry_path,output]);require(not output.exists(),'Immutable annotated output exists')
 before={str(p):sha(p) for p in [video,measurement,native_path,geometry_path]};native=read(native_path);validate_native(native);native_hash=semantic_hash(native);fresh=read(measurement);geometry=read(geometry_path);identity=native['source_identity']
 require(before[str(video)]==identity['video_sha256'] and before[str(measurement)]==identity['records_sha256'] and before[str(geometry_path)]==identity['geometry_sha256'],'Renderer source binding differs')
 require(fresh['records_semantic_sha256']==semantic_hash(fresh['records']),'Measurement integrity failure')
 expected=native['report']['frame_count'];require(fresh['frames']==expected==len(fresh['records']),'Renderer coverage differs');count=expected if preview_limit is None else min(preview_limit,expected)
 geometry['_class_names']=fresh['class_names'];p=plan(native);cap=cv2.VideoCapture(str(video));require(cap.isOpened(),'Source cannot decode');fps=cap.get(cv2.CAP_PROP_FPS);require(math.isfinite(fps) and fps>0,'Invalid FPS')
 w,h=identity['width'],identity['height'];size=(w+560,max(900,h));require(size[0]%2==0 and size[1]%2==0,'H264 requires even output dimensions');output.parent.mkdir(parents=True,exist_ok=True)
 temp=output.with_name(output.stem+'.partial.mp4');log=output.with_suffix('.ffmpeg.log');command=['ffmpeg','-hide_banner','-loglevel','error','-f','rawvideo','-pix_fmt','bgr24','-s',f'{size[0]}x{size[1]}','-r',str(fps),'-i','pipe:0','-an','-c:v','libx264','-threads','2','-preset','veryfast','-crf','23','-pix_fmt','yuv420p','-movflags','+faststart',str(temp)]
 encoded=0
 try:
  with log.open('w') as stderr:
   proc=subprocess.Popen(command,stdin=subprocess.PIPE,stderr=stderr)
   try:
    for index,record in enumerate(fresh['records'][:count]):
     ok,image=cap.read();require(ok and record['frame_index']==index,'Decode/frame gap')
     require(hashlib.sha256(image.tobytes()).hexdigest()==record['source_pixel_sha256'],'Source pixel mismatch: sequential decode required')
     canvas,state=paint(image,record,native,geometry,p,fps=fps,video_id=native['video_id']);proc.stdin.write(canvas.tobytes());encoded+=1
     if index%300==0:print(json.dumps({'stage':'annotation','video':native['video_id'],'frame':index,'total':count}),flush=True)
   finally:proc.stdin.close()
   require(proc.wait(timeout=180)==0,'FFmpeg failed; see codec log')
  require(encoded==count,'Incomplete annotated video')
  if preview_limit is None:require(not cap.read()[0],'Unexpected extra source frames; not a full-video render')
  if preview_limit is None:require(state['accepted']==native['aggregate']['total'] and state['vehicles']==native['aggregate']['vehicle_total'],'Final panel/ledger mismatch')
  dec=cv2.VideoCapture(str(temp));decoded=0
  while True:
   ok,_=dec.read()
   if not ok:break
   decoded+=1
  dec.release();require(decoded==count,'Encoded frame coverage mismatch')
  require(semantic_hash(native)==native_hash and all(sha(Path(k))==v for k,v in before.items()),'Renderer changed inputs')
  temp.rename(output);report={'status':'SUCCEEDED','video_id':native['video_id'],'frames_encoded':encoded,'frames_redecoded':decoded,'measurement_frames':expected,'source_reported_frames':fresh['metadata']['reported_frame_count'],'scope':'FULL_VIDEO' if preview_limit is None else 'PREVIEW_ONLY','final_panel':{k:v for k,v in state.items() if k!='recent'},'output_sha256':sha(output),'input_sha256':before,'counting_rerun':False,'inference_rerun':False,'source_pixels_verified_every_frame':True,'audio_preserved':False,'timing':'CFR_REPORTED_FPS_NOT_ORIGINAL_VFR_PTS'};write(output.with_suffix('.json'),report);return report
 except BaseException:
  if 'proc' in locals() and proc.poll() is None:proc.kill();proc.wait()
  temp.unlink(missing_ok=True);raise
 finally:cap.release()


## src/traffic_location/traffic/_kernel/__init__.py

In [ ]:
%%writefile /kaggle/working/t7sl_annotation_source/src/traffic_location/traffic/_kernel/__init__.py
"""Independent engine components; legacy algorithm semantics are preserved."""


## src/traffic_location/traffic/_kernel/geometry.py

In [ ]:
%%writefile /kaggle/working/t7sl_annotation_source/src/traffic_location/traffic/_kernel/geometry.py
"""Phase 5 notebook geometry, independent of detector/tracker/model quality.

Outputs are geometric diagnostics, never accepted traffic-count events. All
noise/confirmation parameters are required experimental inputs. ``neutral_band_px``
is the distance on EACH side of the line, not the full width of that band.

Only floating-point roundoff is tolerated by primitives (1e-12 times coordinate
scale); that numerical tolerance is not a jitter/counting threshold.
"""
from __future__ import annotations
from collections import Counter
from dataclasses import dataclass
import gzip
import hashlib
import json
import math
from numbers import Real
from pathlib import Path
from typing import Any, Iterable
Point = tuple[float, float]

def _point(value: Iterable[float]) -> Point:
    try:
        values = tuple(value)
    except TypeError as exc:
        raise ValueError('A point requires two finite numeric coordinates') from exc
    if len(values) != 2 or any((isinstance(x, bool) or not isinstance(x, Real) for x in values)):
        raise ValueError('A point requires two finite numeric coordinates')
    try:
        result = (float(values[0]), float(values[1]))
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError('A point requires two finite numeric coordinates') from exc
    if not all((math.isfinite(x) for x in result)):
        raise ValueError('Non-finite geometry coordinate')
    return result

def _epsilon(*points: Point) -> float:
    return 1e-12 * max(1.0, *(abs(v) for p in points for v in p))

def _cross(a: Point, b: Point) -> float:
    return a[0] * b[1] - a[1] * b[0]

def _subtract(a: Point, b: Point) -> Point:
    return (a[0] - b[0], a[1] - b[1])

def _distance(a: Point, b: Point) -> float:
    return math.hypot(a[0] - b[0], a[1] - b[1])

def _on_segment(point: Point, a: Point, b: Point) -> bool:
    tolerance = _epsilon(point, a, b)
    length = _distance(a, b)
    if length <= tolerance:
        return _distance(point, a) <= tolerance
    if abs(_cross(_subtract(b, a), _subtract(point, a))) / length > tolerance:
        return False
    return min(a[0], b[0]) - tolerance <= point[0] <= max(a[0], b[0]) + tolerance and min(a[1], b[1]) - tolerance <= point[1] <= max(a[1], b[1]) + tolerance

def point_in_polygon(point: Iterable[float], polygon: Iterable[Iterable[float]]) -> str:
    """Return inside/boundary/outside; no implicit pixel or jitter margin."""
    point = _point(point)
    vertices = tuple((_point(p) for p in polygon))
    if len(vertices) < 3:
        raise ValueError('A polygon needs at least three vertices')
    inside = False
    for a, b in zip(vertices, vertices[1:] + vertices[:1]):
        if _on_segment(point, a, b):
            return 'boundary'
        if (a[1] > point[1]) != (b[1] > point[1]):
            intersection_x = a[0] + (point[1] - a[1]) * (b[0] - a[0]) / (b[1] - a[1])
            if point[0] < intersection_x:
                inside = not inside
    return 'inside' if inside else 'outside'

def signed_distance(point: Iterable[float], p1: Iterable[float], p2: Iterable[float]) -> float:
    """Signed perpendicular distance in pixels; sign has no A/B semantics."""
    point, p1, p2 = (_point(point), _point(p1), _point(p2))
    length = _distance(p1, p2)
    if length <= _epsilon(p1, p2):
        raise ValueError('Counting-line endpoints must be distinct')
    return _cross(_subtract(p2, p1), _subtract(point, p1)) / length

def segment_intersection(a, b, c, d) -> dict[str, Any]:
    """Intersect finite path a-b with finite line c-d.

    ``proper``: interior of both segments. ``endpoint``: configured-line c/d
    contact. ``touch``: path endpoint at line interior. ``overlap``: positive
    collinear overlap. ``none`` includes intersections of infinite extensions.
    ``t`` and ``u`` are path and counting-line fractions, respectively.
    """
    a, b, c, d = (_point(a), _point(b), _point(c), _point(d))
    r, s = (_subtract(b, a), _subtract(d, c))
    r_length, s_length = (_distance(a, b), _distance(c, d))
    tolerance = _epsilon(a, b, c, d)
    empty = {'kind': 'none', 'point': None, 't': None, 'u': None}
    if s_length <= tolerance:
        raise ValueError('Configured line must have distinct endpoints')
    if r_length <= tolerance:
        if not _on_segment(a, c, d):
            return empty
        u = sum(((a[i] - c[i]) * s[i] for i in (0, 1))) / (s_length * s_length)
        endpoint = _distance(a, c) <= tolerance or _distance(a, d) <= tolerance
        return {'kind': 'endpoint' if endpoint else 'touch', 'point': list(a), 't': 0.0, 'u': u}
    denominator = _cross(r, s)
    delta = _subtract(c, a)
    if abs(denominator) <= tolerance * max(r_length, s_length):
        if abs(_cross(delta, r)) / r_length > tolerance:
            return empty
        t_c = sum((delta[i] * r[i] for i in (0, 1))) / (r_length * r_length)
        t_d = sum(((d[i] - a[i]) * r[i] for i in (0, 1))) / (r_length * r_length)
        lower, upper = (max(0.0, min(t_c, t_d)), min(1.0, max(t_c, t_d)))
        fraction_tolerance = tolerance / r_length
        if lower > upper + fraction_tolerance:
            return empty
        point = (a[0] + lower * r[0], a[1] + lower * r[1])
        if upper - lower > fraction_tolerance:
            end = [a[0] + upper * r[0], a[1] + upper * r[1]]
            return {'kind': 'overlap', 'point': list(point), 'end_point': end, 't': lower, 't_end': upper, 'u': None}
        endpoint = _distance(point, c) <= tolerance or _distance(point, d) <= tolerance
        u = sum(((point[i] - c[i]) * s[i] for i in (0, 1))) / (s_length * s_length)
        return {'kind': 'endpoint' if endpoint else 'touch', 'point': list(point), 't': lower, 'u': u}
    t, u = (_cross(delta, s) / denominator, _cross(delta, r) / denominator)
    t_tolerance, u_tolerance = (tolerance / r_length, tolerance / s_length)
    if not (-t_tolerance <= t <= 1 + t_tolerance and -u_tolerance <= u <= 1 + u_tolerance):
        return empty
    t, u = (min(1.0, max(0.0, t)), min(1.0, max(0.0, u)))
    point = [a[0] + t * r[0], a[1] + t * r[1]]
    endpoint = u <= u_tolerance or u >= 1 - u_tolerance
    touch = t <= t_tolerance or t >= 1 - t_tolerance
    return {'kind': 'endpoint' if endpoint else 'touch' if touch else 'proper', 'point': point, 't': t, 'u': u}

def _validate_polygon(vertices: tuple[Point, ...], name: str) -> None:
    if len(vertices) < 3:
        raise ValueError(f'{name} requires at least three vertices')
    edges = list(zip(vertices, vertices[1:] + vertices[:1]))
    if any((_distance(a, b) <= _epsilon(a, b) for a, b in edges)):
        raise ValueError(f'{name} has a zero-length edge')
    twice_area = sum((_cross(a, b) for a, b in edges))
    if abs(twice_area) <= _epsilon(*vertices) * sum((_distance(a, b) for a, b in edges)):
        raise ValueError(f'{name} has zero area')
    for i, (a, b) in enumerate(edges):
        for j, (c, d) in enumerate(edges):
            if j <= i or j == i + 1 or (i == 0 and j == len(edges) - 1):
                continue
            if segment_intersection(a, b, c, d)['kind'] != 'none':
                raise ValueError(f'{name} is not a simple polygon')

def _interior_samples(vertices: tuple[Point, ...]):
    """Interior points from horizontal scan intervals, including concave shapes.

    Used only for topology validation, not placement/noise thresholds. Samples
    are strict interior interval midpoints rather than a possibly exterior
    centroid. They detect coincident outlines with different vertex sampling.
    """
    levels = sorted(set((p[1] for p in vertices)))
    for lower, upper in zip(levels, levels[1:]):
        y = (lower + upper) / 2
        intersections = []
        for a, b in zip(vertices, vertices[1:] + vertices[:1]):
            if (a[1] > y) != (b[1] > y):
                intersections.append(a[0] + (y - a[1]) * (b[0] - a[0]) / (b[1] - a[1]))
        intersections.sort()
        for left, right in zip(intersections[::2], intersections[1::2]):
            point = ((left + right) / 2, y)
            if point_in_polygon(point, vertices) == 'inside':
                yield point

@dataclass(frozen=True)
class Geometry:
    p1: Point
    p2: Point
    side_a: tuple[Point, ...]
    side_b: tuple[Point, ...]
    width: int | None = None
    height: int | None = None
    geometry_id: str = 'explicit-fixture'
    video_sha256: str | None = None
    source_draft_status: str | None = None

    def __post_init__(self):
        for name in ('p1', 'p2'):
            object.__setattr__(self, name, _point(getattr(self, name)))
        for name in ('side_a', 'side_b'):
            vertices = tuple((_point(p) for p in getattr(self, name)))
            _validate_polygon(vertices, name)
            object.__setattr__(self, name, vertices)
        signed_distance(self.p1, self.p1, self.p2)
        if (self.width is None) != (self.height is None):
            raise ValueError('Both width and height must be provided, or neither')
        if self.width is not None:
            if any((isinstance(v, bool) or not isinstance(v, int) or v <= 0 for v in (self.width, self.height))):
                raise ValueError('Native dimensions must be positive integers')
            for x, y in (self.p1, self.p2, *self.side_a, *self.side_b):
                if not (0 <= x <= self.width and 0 <= y <= self.height):
                    raise ValueError('Geometry exceeds declared continuous image-edge coordinates')
        for first, second in ((self.side_a, self.side_b), (self.side_b, self.side_a)):
            if any((point_in_polygon(p, second) == 'inside' for p in _interior_samples(first))):
                raise ValueError('Side A/B polygon interiors overlap')
            for a, b in zip(first, first[1:] + first[:1]):
                midpoint = ((a[0] + b[0]) / 2, (a[1] + b[1]) / 2)
                if any((point_in_polygon(p, second) == 'inside' for p in (a, midpoint))):
                    raise ValueError('Side A/B polygon interiors overlap')
                for c, d in zip(second, second[1:] + second[:1]):
                    if segment_intersection(a, b, c, d)['kind'] == 'proper':
                        raise ValueError('Side A/B polygon interiors overlap')

    @classmethod
    def from_draft(cls, path, width: int, height: int) -> Geometry:
        """Resolve recorded normalized coordinates without snapping/cropping."""
        draft = json.loads(Path(path).read_text(encoding='utf-8'))
        normalized = draft['normalized']

        def resolve(p):
            u, v = _point(p)
            if not (0 <= u <= 1 and 0 <= v <= 1):
                raise ValueError('Normalized template coordinate outside [0,1]')
            return (u * width, v * height)
        return cls(resolve(normalized['line']['P1']), resolve(normalized['line']['P2']), tuple((resolve(p) for p in normalized['side_a_polygon'])), tuple((resolve(p) for p in normalized['side_b_polygon'])), width, height, draft.get('draft_id', 'explicit-template'), draft.get('source_video', {}).get('sha256'), draft.get('status'))

    def side(self, point: Iterable[float]) -> str | None:
        states = (point_in_polygon(point, self.side_a), point_in_polygon(point, self.side_b))
        if 'boundary' in states:
            return None
        if states == ('inside', 'outside'):
            return 'A'
        if states == ('outside', 'inside'):
            return 'B'
        return None

    def as_dict(self) -> dict[str, Any]:
        return {'geometry_id': self.geometry_id, 'width': self.width, 'height': self.height, 'coordinate_space': 'native decoded-frame continuous image-edge pixels', 'line': {'P1': list(self.p1), 'P2': list(self.p2)}, 'side_a_polygon': [list(p) for p in self.side_a], 'side_b_polygon': [list(p) for p in self.side_b], 'video_sha256': self.video_sha256, 'source_draft_status': self.source_draft_status}

class GeometryStateMachine:
    """One track's measured trajectory, with no category/counting policy.

    Origin and destination require polygon interior and distance outside the
    experimental neutral band. Missing/predicted observations never bridge a
    passage. A successful transition rebases geometric origin, so future return
    diagnostics remain available while their business counting policy stays open.
    """

    def __init__(self, geometry: Geometry, *, neutral_band_px: float, confirmation_observations: int):
        if isinstance(neutral_band_px, bool) or not isinstance(neutral_band_px, Real) or (not math.isfinite(neutral_band_px)) or (neutral_band_px < 0):
            raise ValueError('neutral_band_px must be an explicit finite nonnegative value')
        if isinstance(confirmation_observations, bool) or not isinstance(confirmation_observations, int) or confirmation_observations < 1:
            raise ValueError('confirmation_observations must be an explicit positive integer')
        self.geometry = geometry
        self.neutral_band_px = float(neutral_band_px)
        self.confirmation_observations = confirmation_observations
        self.last_update_frame = None
        self.last_observed = None
        self.origin = None
        self.pending = None
        self.touch_sequence = None
        self.continuity_broken = False
        self.transition_sequence = 0

    def _observation(self, frame_index, point, metadata):
        distance = signed_distance(point, self.geometry.p1, self.geometry.p2)
        zero = abs(distance) <= _epsilon(point, self.geometry.p1, self.geometry.p2)
        sign = 0 if zero else 1 if distance > 0 else -1
        side = self.geometry.side(point)
        stable_side = side if abs(distance) > self.neutral_band_px and (not zero) else None
        return {'frame_index': frame_index, 'point': list(point), 'metadata': dict(metadata or {}), 'side': side, 'stable_side': stable_side, 'signed_distance_px': distance, 'sign': sign}

    def _diagnostic(self, kind, **fields):
        return {'type': kind, 'geometry_id': self.geometry.geometry_id, 'accepted_count_event': False, 'repeat_business_policy_open': True, **fields}

    def _establish_origin(self, observation, diagnostics):
        if observation['stable_side'] is not None:
            self.origin = observation
            diagnostics.append(self._diagnostic('ORIGIN_ESTABLISHED', origin_side=observation['stable_side'], origin_frame=observation['frame_index'], origin_point=observation['point'], metadata=observation['metadata']))

    def update(self, frame_index: int, point, *, observed: bool=True, metadata=None) -> list[dict[str, Any]]:
        if isinstance(frame_index, bool) or not isinstance(frame_index, int) or frame_index < 0:
            raise ValueError('frame_index must be a nonnegative integer')
        if self.last_update_frame is not None and frame_index <= self.last_update_frame:
            raise ValueError('Track updates must have strictly increasing frame indices')
        if not isinstance(observed, bool):
            raise ValueError('observed must explicitly be boolean')
        self.last_update_frame = frame_index
        diagnostics = []
        if not observed:
            self.pending = None
            self.touch_sequence = None
            self.continuity_broken = True
            return [self._diagnostic('PREDICTION_IGNORED', frame_index=frame_index, metadata=dict(metadata or {}))]
        point = _point(point)
        current = self._observation(frame_index, point, metadata)
        previous = self.last_observed
        if previous is not None and (frame_index != previous['frame_index'] + 1 or self.continuity_broken):
            intersection = segment_intersection(previous['point'], point, self.geometry.p1, self.geometry.p2)
            possible_passage = self.pending is not None or self.touch_sequence is not None or (self.origin is not None and (previous['stable_side'] != current['stable_side'] or previous['sign'] != current['sign'] or intersection['kind'] != 'none'))
            if possible_passage:
                diagnostics.append(self._diagnostic('AMBIGUOUS_GAP', previous_observation=previous, current_observation=current, missing_observation_frames=frame_index - previous['frame_index'] - 1, interpolation_used=False))
            self.origin = self.pending = self.touch_sequence = None
            self.last_observed = current
            self.continuity_broken = False
            self._establish_origin(current, diagnostics)
            return diagnostics
        self.continuity_broken = False
        if self.origin is None:
            self._establish_origin(current, diagnostics)
        elif previous is not None:
            intersection = segment_intersection(previous['point'], point, self.geometry.p1, self.geometry.p2)
            overlap_reaches_endpoint = intersection['kind'] == 'overlap' and any((_on_segment(endpoint, _point(previous['point']), point) for endpoint in (self.geometry.p1, self.geometry.p2)))
            if intersection['kind'] == 'endpoint' or overlap_reaches_endpoint:
                diagnostics.append(self._diagnostic('AMBIGUOUS_ENDPOINT', previous_observation=previous, current_observation=current, intersection=intersection))
                self.pending = self.touch_sequence = None
            else:
                crossing = None
                if intersection['kind'] == 'proper' and previous['sign'] * current['sign'] == -1:
                    crossing = {'intersection': intersection, 'start_observation': previous, 'end_observation': current, 'method': 'consecutive_measured_segment', 'entry_sign': previous['sign'], 'exit_sign': current['sign']}
                elif current['sign'] == 0 and intersection['kind'] in ('touch', 'overlap'):
                    if self.touch_sequence is None and previous['sign'] != 0:
                        self.touch_sequence = {'entry_sign': previous['sign'], 'start_observation': previous, 'first_touch_observation': current, 'last_touch_observation': current, 'intersection': intersection}
                    elif self.touch_sequence is not None:
                        self.touch_sequence['last_touch_observation'] = current
                elif previous['sign'] == 0 and current['sign'] != 0:
                    if self.touch_sequence is not None and current['sign'] == -self.touch_sequence['entry_sign']:
                        crossing = {'intersection': self.touch_sequence['intersection'], 'start_observation': self.touch_sequence['start_observation'], 'end_observation': current, 'method': 'measured_touch_then_opposite_sign', 'entry_sign': self.touch_sequence['entry_sign'], 'exit_sign': current['sign'], 'first_touch_observation': self.touch_sequence['first_touch_observation'], 'last_touch_observation': self.touch_sequence['last_touch_observation']}
                    self.touch_sequence = None
                if self.pending is not None and current['sign'] == self.pending['crossing']['entry_sign']:
                    self.pending = None
                if crossing is not None:
                    self.pending = {'origin': self.origin, 'crossing': crossing, 'confirmations': [], 'left_origin_region': current['side'] != self.origin['stable_side']}
                elif self.pending is not None:
                    if current['side'] != self.origin['stable_side']:
                        self.pending['left_origin_region'] = True
                    elif current['stable_side'] == self.origin['stable_side'] and self.pending['left_origin_region']:
                        self.pending = None
                if self.pending is not None:
                    destination = 'B' if self.origin['stable_side'] == 'A' else 'A'
                    if current['stable_side'] == destination and current['sign'] == self.pending['crossing']['exit_sign']:
                        self.pending['confirmations'].append(current)
                    else:
                        self.pending['confirmations'] = []
                    if len(self.pending['confirmations']) >= self.confirmation_observations:
                        crossing = self.pending['crossing']
                        self.transition_sequence += 1
                        diagnostics.append(self._diagnostic('GEOMETRY_TRANSITION', software_transition_sequence=self.transition_sequence, origin_side=self.origin['stable_side'], destination_side=destination, direction=f"{self.origin['stable_side']}_TO_{destination}", origin_frame=self.origin['frame_index'], origin_point=self.origin['point'], crossing_frame_start=crossing['start_observation']['frame_index'], crossing_frame_end=crossing['end_observation']['frame_index'], crossing_point=crossing['intersection']['point'], confirmation_frame=frame_index, confirmation_point=current['point'], origin_observation=self.origin, crossing_evidence=crossing, confirmation_observations=self.pending['confirmations'], metadata=current['metadata']))
                        self.origin = current
                        self.pending = self.touch_sequence = None
        self.last_observed = current
        return diagnostics

def replay_cached_records(records_path, geometry: Geometry, *, neutral_band_px: float, confirmation_observations: int) -> dict[str, Any]:
    """Stream existing ReID emitted observations; no model inference or totals.

    Only raw detector-linked boxes of persistent emitted tracks provide BC.
    All raw classes remain diagnostic inputs, not accepted product categories.
    Source links, dimensions, video identity and measurement provenance are
    validated; predictions/lifecycle boxes are never substituted.
    """
    GeometryStateMachine(geometry, neutral_band_px=neutral_band_px, confirmation_observations=confirmation_observations)
    path = Path(records_path)
    source_hash = hashlib.sha256()
    with path.open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            source_hash.update(block)
    opener = gzip.open if path.suffix == '.gz' else open
    machines, diagnostics = ({}, [])
    frame_count = observation_count = 0
    previous_frame = None
    source_video_sha256 = None
    source_parent_run_id = source_variant = None
    with opener(path, 'rt', encoding='utf-8') as source:
        for line_number, line in enumerate(source, 1):
            if not line.strip():
                raise ValueError(f'Empty source record on line {line_number}')
            record = json.loads(line)
            frame = record['frame_index']
            if isinstance(frame, bool) or not isinstance(frame, int) or frame < 0 or (previous_frame is not None and frame <= previous_frame):
                raise ValueError('Cached frame records must be strictly increasing nonnegative indices')
            previous_frame = frame
            frame_count += 1
            if (record['width'], record['height']) != (geometry.width, geometry.height):
                raise ValueError('Cached native dimensions do not match effective geometry')
            video_sha256 = record['video_sha256']
            if source_video_sha256 is None:
                source_video_sha256 = video_sha256
            if video_sha256 != source_video_sha256 or (geometry.video_sha256 is not None and video_sha256 != geometry.video_sha256):
                raise ValueError('Cached source video identity changed or does not match geometry')
            if frame_count == 1:
                source_parent_run_id, source_variant = (record['parent_run_id'], record['variant'])
            elif (record['parent_run_id'], record['variant']) != (source_parent_run_id, source_variant):
                raise ValueError('Cached parent run or variant changed within one replay input')
            raw_detections = record['raw_detections']
            frame_ids, frame_detection_indices = (set(), set())
            for observation in record['emitted_tracks']:
                track_id = observation['track_id']
                if isinstance(track_id, bool) or not isinstance(track_id, int) or track_id < 1 or (track_id in frame_ids):
                    raise ValueError('Emitted track IDs must be positive and unique within a frame')
                frame_ids.add(track_id)
                if observation['frame_index'] != frame or observation.get('emitted') is not True:
                    raise ValueError('Emitted measurement/frame contract mismatch')
                if observation.get('measurement_provenance') != 'retained_phase_2_detection':
                    raise ValueError('Geometry requires retained measured detection provenance')
                index = observation['original_detection_index']
                if isinstance(index, bool) or not isinstance(index, int) or (not 0 <= index < len(raw_detections)):
                    raise ValueError('Broken source-detection index')
                if index in frame_detection_indices:
                    raise ValueError('One source detection cannot supply two emitted track measurements in the same frame')
                frame_detection_indices.add(index)
                raw_bbox = observation['raw_bbox']
                if len(raw_bbox) != 4:
                    raise ValueError('Raw bbox must have four xyxy coordinates')
                a, b = (_point(raw_bbox[:2]), _point(raw_bbox[2:]))
                if a[0] >= b[0] or a[1] >= b[1]:
                    raise ValueError('Raw bbox must have positive width/height')
                detection = raw_detections[index]
                for confidence in (observation['confidence'], detection['confidence']):
                    if isinstance(confidence, bool) or not isinstance(confidence, Real) or (not math.isfinite(confidence)):
                        raise ValueError('Source and observation confidence must be finite numeric measurements')
                if list(raw_bbox) != detection['bbox_xyxy_pixels']:
                    raise ValueError('Raw bbox is not byte-value linked to source detection')
                if any((observation[key] != detection[key] for key in ('class_id', 'class_name', 'confidence'))):
                    raise ValueError('Track measurement class/confidence does not match linked source')
                metadata = {'track_id': track_id, 'video_sha256': video_sha256, 'parent_run_id': record['parent_run_id'], 'variant': record['variant'], 'original_detection_index': index, 'raw_bbox': list(raw_bbox), 'class_id': observation['class_id'], 'class_name': observation['class_name'], 'confidence': observation['confidence'], 'measurement_provenance': observation['measurement_provenance'], 'timestamp_estimate_from_index_fps_seconds': record.get('timestamp_estimate_from_index_fps_seconds'), 'timestamp_is_verified_source_pts': False}
                machine = machines.setdefault(track_id, GeometryStateMachine(geometry, neutral_band_px=neutral_band_px, confirmation_observations=confirmation_observations))
                anchor = ((a[0] + b[0]) / 2, b[1])
                for diagnostic in machine.update(frame, anchor, metadata=metadata):
                    diagnostics.append({'track_id': track_id, 'frame_index': frame, **diagnostic})
                observation_count += 1
    return {'diagnostics': diagnostics, 'report': {'claim_type': 'EXECUTED_SOFTWARE_GEOMETRY_DIAGNOSTICS_NOT_TRAFFIC_COUNTS', 'source_records_path': str(path), 'source_records_file_sha256': source_hash.hexdigest(), 'video_sha256': source_video_sha256, 'source_parent_run_id': source_parent_run_id, 'source_variant': source_variant, 'frames_processed': frame_count, 'emitted_measured_observations_processed': observation_count, 'software_track_ids': len(machines), 'software_diagnostic_counts': dict(Counter((d['type'] for d in diagnostics))), 'effective_geometry': geometry.as_dict(), 'anchor': 'raw_bbox_bottom_center', 'neutral_band_px': float(neutral_band_px), 'neutral_band_semantics': 'half-width per side', 'confirmation_observations': confirmation_observations, 'category_filter': 'none_all_raw_classes_are_diagnostic_inputs_only', 'detector_tracker_encoder_rerun': False, 'accepted_count_events_emitted': False, 'physical_identity_or_counting_accuracy_established': False, 'repeat_business_policy': 'OPEN', 'gap_policy': 'no interpolation; ambiguous passage; origin reset'}}


## tools/render_t7sl_results.py

In [ ]:
%%writefile /kaggle/working/t7sl_annotation_source/tools/render_t7sl_results.py
"""Render completed producer outputs. No detector/tracker/ReID/counting execution."""
from pathlib import Path
import json,gzip,shutil,concurrent.futures
from traffic_location.artifacts import read,write,sha,semantic_hash,require
from traffic_location.traffic.annotation import render

def main(producer,video_root,output):
 producer,video_root,output=map(Path,[producer,video_root,output]);require(not output.exists(),'Annotation revision output exists');output.mkdir(parents=True)
 combined=read(producer/'combined.json');jobs=read(producer/'jobs.json');require(combined['videos_completed']==8 and len(jobs)==8 and all(j['status']=='SUCCEEDED' for j in jobs),'Eight full-video results required')
 original_hash=semantic_hash(combined);shutil.copyfile(producer/'combined.json',output/'combined.json');shutil.copyfile(producer/'combined.csv',output/'combined.csv');shutil.copyfile(producer/'jobs.json',output/'jobs.json')
 def artifact(path):
  if path.exists():return path
  if path.suffix=='.gz' and path.with_suffix('').exists():return path.with_suffix('')
  raise ValueError('Missing producer artifact: '+str(path))
 def run(j):
  vid=j['video_id'];job=producer/'videos'/vid;native_path=artifact(job/'counting/native.json.gz');native=read(native_path);wrapper=read(job/'counting/result.json')
  require(semantic_hash(native)==wrapper['native_semantic_sha256'],'Native result semantic hash differs')
  measurement=artifact(job/'measurement/result.json.gz')
  if sha(measurement)!=native['source_identity']['records_sha256']:
   # Kaggle can expand gzip attachments. Reconstruct exact gzip from the original
   # JSON bytes in the pinned Python/zlib runtime, then require the original SHA.
   recovered=output/'recovered'/vid/'result.json.gz';recovered.parent.mkdir(parents=True,exist_ok=True);recovered.write_bytes(gzip.compress(measurement.read_bytes(),mtime=0))
   require(sha(recovered)==native['source_identity']['records_sha256'],'Original measurement bytes cannot be restored; do not weaken identity checks');measurement=recovered
  report=render(video_root/(vid+'.mp4'),measurement,native_path,job/'geometry.json',output/'annotated'/(vid+'.mp4'))
  require(report['frames_encoded']==j['frames'] and report['scope']=='FULL_VIDEO','Annotation incomplete')
  return report
 with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:reports=list(pool.map(run,jobs))
 require(semantic_hash(read(producer/'combined.json'))==original_hash,'Producer result changed')
 write(output/'annotation_manifest.json',{'status':'SUCCEEDED','videos':reports,'frames':sum(r['frames_encoded'] for r in reports),'inference_rerun':False,'counting_rerun':False,'source_combined_semantic_sha256':original_hash})
 import html
 cards=''.join('<h2>'+html.escape(r['video_id'])+'</h2><p>'+str(r['frames_encoded'])+' frames</p><video controls preload="none" width="960" src="annotated/'+html.escape(r['video_id'])+'.mp4"></video>' for r in reports)
 (output/'review.html').write_text('<!doctype html><meta charset="utf-8"><title>T7SL annotated review</title><h1>Full-frame offline review</h1><p>Geometry provisional; measured and estimated separate. Source counts unchanged.</p>'+cards)
 return reports


## Validate completed input results and full-source videos

In [ ]:
sys.path.insert(0,str(ROOT/'src'));sys.path.insert(0,str(ROOT))
from traffic_location.artifacts import read,sha
roots=[p.parent for p in Path('/kaggle/input').rglob('combined.json') if (p.parent/'jobs.json').exists()]
assert len(roots)==1,roots
PRODUCER=roots[0];summary=read(PRODUCER/'combined.json');assert summary['videos_completed']==8
hits=list(Path('/kaggle/input').rglob('T7SL-WEEKDAY-PAGI.mp4'));assert len(hits)==1;VIDEO_ROOT=hits[0].parent
print('Producer:',PRODUCER,'Videos:',summary['videos_completed'])
print('Video source:',VIDEO_ROOT)
subprocess.run(['ffmpeg','-version'],check=True)


## Render all source frames and verify output
MP4 includes raw IDs, category colors and label backgrounds, exact-frame rider union with parent raw BC, geometry+margin, category/direction pivot, accepted passage logs, measured/estimated and final pending/invalid. Panel is explicitly offline finalized. No accuracy claim.

In [ ]:
import cv2
cv2.setNumThreads(1)
from tools.render_t7sl_results import main
OUTPUT=Path('/kaggle/working/results/t7sl_annotated_full_v1')
reports=main(PRODUCER,VIDEO_ROOT,OUTPUT)
print('ALL ANNOTATED:',len(reports),'frames:',sum(r['frames_encoded'] for r in reports))


## Open output directly from Kaggle
Open Output → results/t7sl_annotated_full_v1/annotated/. Each MP4 has its own JSON audit. combined.csv/json preserve the producer counts. review.html is a standalone video index.

In [ ]:
import pandas as pd
from IPython.display import display,FileLink
display(pd.read_csv(OUTPUT/'combined.csv'))
for report in reports:
 print(report['video_id'],report['frames_encoded'],'frames verified')
 display(FileLink(str(OUTPUT/'annotated'/(report['video_id']+'.mp4'))))
